In [1]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import glob
import os
from hta.trace_analysis import TraceAnalysis
import math

In [2]:
import subprocess
import polars as pl
import re

def demangle_name(
    df: pl.DataFrame,
    mangled_col: str,
    new_col_name: str = "demangled_name"
) -> pl.DataFrame:
    if mangled_col not in df.columns:
        raise ValueError(f"Column '{mangled_col}' not found.")

    mangled_names = df[mangled_col].to_list()
    if not mangled_names:
        return df.with_columns(pl.lit(None, dtype=pl.String).alias(new_col_name))

    input_text = "\n".join(mangled_names)
    
    # ADDED FLAG: -p (no-params)
    # This keeps templates <...> but removes function args (...)
    # This helps significantly because Kineto and AccelSim often differ wildly on function args
    command = ['/usr/local/cuda/bin/cu++filt', '-p'] 
    
    try:
        result = subprocess.run(
            command,
            input=input_text,
            capture_output=True,
            text=True,
            check=True
        )
        demangled_names = result.stdout.strip().split('\n')
    except Exception as e:
        print(f"Demangling failed: {e}")
        return df.with_columns(pl.col(mangled_col).alias(new_col_name))

    if len(mangled_names) != len(demangled_names):
         # Fallback if counts mismatch (rare)
        return df.with_columns(pl.col(mangled_col).alias(new_col_name))

    return df.with_columns(pl.Series(name=new_col_name, values=demangled_names))

In [7]:
def normalize_kernel_name(name: str) -> str:
    if name is None:
        return ""
    
    # 1. Remove all spaces (fixes "char *" vs "char*")
    s = name.replace(" ", "")
    
    # 2. Standardize namespaces
    # AccelSim often uses <unnamed>, Kineto uses (anonymous namespace) or vice versa
    s = s.replace("(anonymousnamespace)", "<unnamed>")
    s = s.replace("::__1::", "::") # Remove libc++ versioning inline
    
    # 3. Remove C-style casts inside templates
    # Replaces "(int)4" with "4", "(unsignedlong)1" with "1"
    s = re.sub(r'\((?:int|long|unsigned|unsignedlong|unsignedint)\)', '', s)
    
    # 4. Remove numeric suffixes
    # Replaces "1ul" or "4L" with "1" or "4"
    # Look for digits followed by U, L, UL, LL (case insensitive) inside template brackets? 
    # To be safe, we just strip common C++ literal suffixes attached to numbers
    s = re.sub(r'(\d+)[uU]?[lL]+', r'\1', s)
    
    # 5. Clean up std:: string variations
    s = s.replace("std::basic_string<char>", "std::string")
    
    return s

# Helper to apply to a Polars DF
def apply_normalization(df: pl.DataFrame, col_name: str) -> pl.DataFrame:
    return df.with_columns(
        pl.col(col_name).map_elements(normalize_kernel_name, return_dtype=pl.String).alias("clean_name")
    )

In [10]:
df_accel = pl.read_csv("/users/zarand1a/accel-sim-framework/kernel_times2.csv").select(
    pl.col("benchmark").alias("model"),
    pl.col("kernel").str.strip_chars().alias("kernel_mangled_col_name"),
    (pl.col("cycle") / 1132).alias("accel_time_us"),
)

In [ ]:
df_trace = demangle_name(df_accel, "kernel_mangled_col_name", "kernel")

In [12]:
df_trace

model,kernel_mangled_col_name,accel_time_us,kernel
str,str,f64,str
"""bert""","""_ZN2at6native44_GLOBAL__N__50c…",9.166078,"""at::native::<unnamed>::indexSe…"
"""bert""","""_ZN2at6native44_GLOBAL__N__50c…",7.567138,"""at::native::<unnamed>::indexSe…"
"""bert""","""_ZN2at6native29vectorized_elem…",5.09894,"""at::native::vectorized_element…"
"""bert""","""_ZN2at6native44_GLOBAL__N__50c…",9.189046,"""at::native::<unnamed>::indexSe…"
"""bert""","""_ZN2at6native29vectorized_elem…",5.09894,"""at::native::vectorized_element…"
…,…,…,…
"""resnet50-train""","""_ZN2at6native57_GLOBAL__N__e65…",21.024735,"""at::native::<unnamed>::multi_t…"
"""resnet50-train""","""_ZN2at6native54_GLOBAL__N__f8a…",44.819788,"""at::native::<unnamed>::multi_t…"
"""resnet50-train""","""_ZN2at6native54_GLOBAL__N__f8a…",152.175795,"""at::native::<unnamed>::multi_t…"


In [ ]:
import os
import glob
import polars as pl
import pandas as pd # Needed for the ExcelWriter engine
from hta.trace_analysis import TraceAnalysis

def get_raw_model_data(kineto_path):
    # 1. Determine Model Names
    model_dir_name = os.path.basename(kineto_path.rstrip('/'))
    
    if "inference" in model_dir_name:
        accel_model_name = model_dir_name.split('-')[0]
    else:
        accel_model_name = model_dir_name
    
    print(f"Processing: {model_dir_name}...")

    # 2. Load Kineto (Trace) Data
    try:
        if not os.path.exists(os.path.join(kineto_path, "device.0.json")):
            print(f"  -> Skipping: device.0.json not found.")
            return None

        analyzer = TraceAnalysis({0: "device.0.json"}, kineto_path)
        rank = 0
        trace_data = analyzer.t.get_trace(rank)
        symbol_table = analyzer.t.symbol_table.get_sym_table()
        
        # Create Kineto DataFrame
        trace_df = pl.concat(
             pl.from_pandas(trace_data[trace_data["stream"] != -1][["name", "dur"]])
             .select(pl.lit(rank).alias("rank"), pl.all())
             for rank, trace_data in analyzer.t.traces.items()
        ).join(
            pl.from_dict({"name": list(range(len(symbol_table))), "s_name": symbol_table}),
            on="name",
        ).filter(
            ~pl.col("s_name").str.starts_with("nccl") & 
            ~pl.col("s_name").is_in(["Stream Sync", "Memcpy DtoD (Device -> Device)", "Memcpy HtoD (Pageable -> Device)", "Memcpy DtoH (Device -> Pinned)"])
        ).select(
            pl.col("s_name").alias("Kineto_Kernel"),
            pl.col("dur").alias("Kineto_Time_us")
        )
        
    except Exception as e:
        print(f"  -> Error reading trace: {e}")
        return None

    return {
        "name": model_dir_name,
        "trace_df": trace_df
    }



In [ ]:
# --- 1. Demangle ---
# Note: Ensure accel_df has the mangled name column. 
# If accel-sim CSV already has a "kernel_demangled" column, use that but run normalization on it.
# Assuming we are working from raw names:
df_accel = demangle_name(df_accel, "kernel_mangled_col_name", "kernel")
df_trace = demangle_name(df_trace, "name", "kernel") # Kineto usually gives mangled names in 'name'

# --- 2. Normalize ---
df_accel = apply_normalization(df_accel, "kernel")
df_trace = apply_normalization(df_trace, "kernel")

# --- 3. Filter ---
# Remove "noisy" kernels that typically don't match (Memcpy, Sync, etc)
# You already do this partially, but ensure it's done before matching
ignored_patterns = ["Memcpy", "memset", "Sync", "nccl"]
df_accel_clean = df_accel.filter(~pl.col("clean_name").str.contains_any(ignored_patterns))
df_trace_clean = df_trace.filter(~pl.col("clean_name").str.contains_any(ignored_patterns))

# --- 4. Pattern Match (Sequence Alignment) ---
# Instead of strict serial index, we align by clean_name.
# Since traces can have missing/extra kernels, we can't use a simple Join.
# We create a "match_id" based on the sequence of appearance per kernel type.

# Add a cumulative count per kernel name to handle recurring kernels
df_accel_clean = df_accel_clean.with_columns(
    pl.col("clean_name").cum_count().over("clean_name").alias("instance_id")
)

df_trace_clean = df_trace_clean.with_columns(
    pl.col("clean_name").cum_count().over("clean_name").alias("instance_id")
)

# --- 5. Join ---
matched_df = df_trace_clean.join(
    df_accel_clean,
    on=["clean_name", "instance_id"], 
    how="inner"
)

print(f"Matched {len(matched_df)} kernels out of {len(df_trace_clean)} trace kernels.")

# --- 6. Plot ---
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.scatter(matched_df["trace_time_us"], matched_df["accel_time_us"], alpha=0.5, s=15)
plt.plot([0, matched_df["trace_time_us"].max()], [0, matched_df["trace_time_us"].max()], 'r--')
plt.xlabel("Kineto Time (us)")
plt.ylabel("Accel-Sim Time (us)")
plt.title("Matched Kernels (Normalized Name Matching)")
plt.grid(True)
plt.show()